https://www-m3.ma.tum.de/bornemann/challengebook/The%20Story/siam100digits_transparencies.pdf

# Implementation notes 
To address the chaotic nature of this system, we need to move away from standard floating-point numbers (`float`), which use 64-bit binary representation (IEEE 754) and are prone to cumulative rounding errors.

I will update the code to use Python's **`decimal`** module. This allows for **arbitrary-precision arithmetic**. By setting `getcontext().prec`, we can control exactly how many significant digits are used throughout the calculation (e.g., 50, 100, or more).

I'll also refine the collision detection logic to ensure that "surface grazing" or "leaving a mirror" doesn't trigger false collisions due to precision limits.


http://googleusercontent.com/immersive_entry_chip/0

# Key Changes for Accuracy:
1.  **Decimal Class**: We replaced all `float` variables with `Decimal`. Unlike floats, `Decimal` can represent numbers like $0.1$ exactly and maintains a user-defined number of significant digits.
2.  **Context Precision**: `getcontext().prec = 60` sets the calculation to 60 digits of precision. Because this is a billiard-style problem (highly sensitive to initial conditions), small errors in the reflection angle grow exponentially. High precision keeps the "true" path stable for much longer.
3.  **Improved Epsilon**: In floating point, an epsilon like `1e-9` is common. With `Decimal`, we can use a much smaller `1e-30` to ensure that we don't accidentally "re-collide" with the same mirror we are currently touching due to precision jitter.
4.  **Exact Inputs**: Initial coordinates are defined as strings (e.g., `D('0.1')`). This is crucial because `Decimal(0.1)` would still inherit the inaccuracy of the float representation of 0.1.

You can increase `getcontext().prec` to `100` or higher if you want to verify that the result has converged. In chaotic systems, if the result changes significantly when you go from 50 to 100 digits, it means 50 digits was not enough.

To accurately represent the precision of IEEE 754 floating-point formats in decimal digits, we look at the **significand (mantissa) bits**. 

Here is the breakdown for the formats you requested:

| Format | Total Bits | Mantissa Bits (incl. implicit) | Decimal Digits (Approx.) |
| :--- | :--- | :--- | :--- |
| **Half Precision** | 16-bit | 11 bits | **3.3** |
| **Single Precision** | 32-bit | 24 bits | **7.2** |
| **Double Precision** | 64-bit | 53 bits | **15.9** |
| **x86 Extended** | 80-bit | 64 bits | **19.2** |

### What this means for your simulation:
* **16-bit:** Will fail almost immediately. It can't even represent $0.1$ or $0.5$ with enough accuracy to calculate a single bounce correctly.
* **32-bit (7 digits):** The "standard" float. In a chaotic system like this, you will likely see the path diverge from the "true" path after only 2 or 3 bounces.
* **64-bit (15-16 digits):** The Python default `float`. This is usually "good enough" for short simulations, but for $t=10$ and a radius of $1/3$, the cumulative error in the reflection angles will make the final position unreliable.
* **80-bit (19 digits):** Often used internally by CPUs (like the old Intel x87 registers) to reduce intermediate rounding errors.

Since your simulation uses `getcontext().prec = 60`, you are currently running at roughly **triple the precision** of a standard 64-bit float, which is excellent for checking the stability of the result.

In [6]:
import math
from decimal import Decimal, getcontext
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ipywidgets import interact, IntSlider

def run_simulation(precision):
    # Set the global precision based on slider input
    getcontext().prec = precision
    
    # Initial parameters converted to Decimal
    D = Decimal
    start_x, start_y = D('0.5'), D('0.1')
    x, y = start_x, start_y
    vx, vy = D('1.0'), D('0.0')  # Heading due east at speed 1
    radius = D('1') / D('3')
    total_time = D('10.0')
    current_time = D('0')

    # For plotting (convert back to float for matplotlib)
    path_points = [(float(x), float(y))]
    visited_mirrors = set()

    # Safety counter
    max_bounces = 500
    bounces = 0

    while current_time < total_time and bounces < max_bounces:
        remaining_time = total_time - current_time
        first_collision_t = None
        collision_mirror = None

        # Determine a search range
        f_x, f_y = float(x), float(y)
        f_vx, f_vy = float(vx), float(vy)
        f_rem = float(remaining_time)
        
        min_i = math.floor(min(f_x, f_x + f_vx * f_rem) - 1)
        max_i = math.ceil(max(f_x, f_x + f_vx * f_rem) + 1)
        min_j = math.floor(min(f_y, f_y + f_vy * f_rem) - 1)
        max_j = math.ceil(max(f_y, f_y + f_vy * f_rem) + 1)

        for i in range(min_i, max_i + 1):
            for j in range(min_j, max_j + 1):
                Di, Dj = D(i), D(j)
                dx, dy = x - Di, y - Dj
                
                A = vx**2 + vy**2
                B = D('2') * (dx * vx + dy * vy)
                C = dx**2 + dy**2 - radius**2
                
                discriminant = B**2 - D('4') * A * C
                
                if discriminant >= 0:
                    sqrt_disc = discriminant.sqrt()
                    t1 = (-B - sqrt_disc) / (D('2') * A)
                    t2 = (-B + sqrt_disc) / (D('2') * A)
                    
                    # Epsilon scaled by precision
                    epsilon = D('1e-' + str(precision // 2)) 
                    for t in [t1, t2]:
                        if t > epsilon:
                            if first_collision_t is None or t < first_collision_t:
                                first_collision_t = t
                                collision_mirror = (i, j)

        if first_collision_t is not None and first_collision_t <= remaining_time:
            x += vx * first_collision_t
            y += vy * first_collision_t
            current_time += first_collision_t
            path_points.append((float(x), float(y)))
            visited_mirrors.add(collision_mirror)
            
            # Reflection
            mi, mj = collision_mirror
            nx, ny = x - D(mi), y - D(mj)
            n_len = (nx**2 + ny**2).sqrt()
            nx, ny = nx / n_len, ny / n_len
            
            dot = vx * nx + vy * ny
            vx = vx - D('2') * dot * nx
            vy = vy - D('2') * dot * ny
            bounces += 1
        else:
            x += vx * remaining_time
            y += vy * remaining_time
            current_time = total_time
            path_points.append((float(x), float(y)))
            break

    final_dist = (x**2 + y**2).sqrt()

    # --- Plotting ---
    fig, ax = plt.subplots(figsize=(10, 7))
    px, py = zip(*path_points)
    ax.plot(px, py, 'b-', linewidth=1.5, alpha=0.7, label='Photon Path', zorder=3)
    ax.scatter(float(start_x), float(start_y), color='green', s=40, label='Start', zorder=5)
    ax.scatter(float(x), float(y), color='red', s=40, label='End', zorder=5)

    # Relevant mirrors
    margin = 1
    min_vis_x, max_vis_x = math.floor(min(px) - margin), math.ceil(max(px) + margin)
    min_vis_y, max_vis_y = math.floor(min(py) - margin), math.ceil(max(py) + margin)
    f_radius = float(radius)

    for i in range(min_vis_x, max_vis_x + 1):
        for j in range(min_vis_y, max_vis_y + 1):
            is_hit = (i, j) in visited_mirrors
            circle = patches.Circle((i, j), f_radius, color='gray' if not is_hit else 'orange', 
                                    alpha=0.15 if not is_hit else 0.4, zorder=2)
            ax.add_patch(circle)

    ax.set_aspect('equal')
    ax.set_title(f"Precision: {precision} digits | Final Dist: {final_dist:.6f}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(loc='upper left')
    plt.show()

# Setup interactive slider for Jupyter
interact_plot = interact(
    run_simulation, 
    precision=IntSlider(min=2, max=100, step=1, value=60, description='Digits:')
)

interactive(children=(IntSlider(value=60, description='Digits:', min=2), Output()), _dom_classes=('widget-inte…

In [7]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ipywidgets import interact, Dropdown

def run_ieee_simulation(bit_depth):
    # Set the numpy data type based on selection
    if bit_depth == '32-bit (Single)':
        dtype = np.float32
        eps = 1e-6
    else:
        dtype = np.float64
        eps = 1e-13
    
    # Initial parameters using direct IEEE types
    start_x, start_y = dtype(0.5), dtype(0.1)
    x, y = start_x, start_y
    vx, vy = dtype(1.0), dtype(0.0)
    radius = dtype(1.0 / 3.0)
    total_time = dtype(10.0)
    current_time = dtype(0.0)

    path_points = [(float(x), float(y))]
    visited_mirrors = set()
    max_bounces = 500
    bounces = 0

    while current_time < total_time and bounces < max_bounces:
        remaining_time = total_time - current_time
        first_collision_t = None
        collision_mirror = None

        # Search range for lattice points
        f_x, f_y = float(x), float(y)
        f_vx, f_vy = float(vx), float(vy)
        f_rem = float(remaining_time)
        
        min_i = math.floor(min(f_x, f_x + f_vx * f_rem) - 1)
        max_i = math.ceil(max(f_x, f_x + f_vx * f_rem) + 1)
        min_j = math.floor(min(f_y, f_y + f_vy * f_rem) - 1)
        max_j = math.ceil(max(f_y, f_y + f_vy * f_rem) + 1)

        for i in range(min_i, max_i + 1):
            for j in range(min_j, max_j + 1):
                Di, Dj = dtype(i), dtype(j)
                dx, dy = x - Di, y - Dj
                
                # Quadratic coefficients
                A = vx**2 + vy**2
                B = dtype(2.0) * (dx * vx + dy * vy)
                C = dx**2 + dy**2 - radius**2
                
                discriminant = B**2 - dtype(4.0) * A * C
                
                if discriminant >= 0:
                    sqrt_disc = np.sqrt(discriminant)
                    t1 = (-B - sqrt_disc) / (dtype(2.0) * A)
                    t2 = (-B + sqrt_disc) / (dtype(2.0) * A)
                    
                    for t in [t1, t2]:
                        if t > eps:
                            if first_collision_t is None or t < first_collision_t:
                                first_collision_t = t
                                collision_mirror = (i, j)

        if first_collision_t is not None and first_collision_t <= remaining_time:
            x += vx * first_collision_t
            y += vy * first_collision_t
            current_time += first_collision_t
            path_points.append((float(x), float(y)))
            visited_mirrors.add(collision_mirror)
            
            # Reflection
            mi, mj = collision_mirror
            nx, ny = x - dtype(mi), y - dtype(mj)
            n_len = np.sqrt(nx**2 + ny**2)
            nx, ny = nx / n_len, ny / n_len
            
            dot = vx * nx + vy * ny
            vx = vx - dtype(2.0) * dot * nx
            vy = vy - dtype(2.0) * dot * ny
            bounces += 1
        else:
            x += vx * remaining_time
            y += vy * remaining_time
            current_time = total_time
            path_points.append((float(x), float(y)))
            break

    final_dist = np.sqrt(x**2 + y**2)

    # --- Plotting ---
    fig, ax = plt.subplots(figsize=(10, 7))
    px, py = zip(*path_points)
    ax.plot(px, py, 'g-', linewidth=1.5, alpha=0.8, label=f'Path ({bit_depth})', zorder=3)
    ax.scatter(float(start_x), float(start_y), color='blue', s=40, label='Start', zorder=5)
    ax.scatter(float(x), float(y), color='red', s=40, label='End', zorder=5)

    margin = 1
    min_vis_x, max_vis_x = math.floor(min(px) - margin), math.ceil(max(px) + margin)
    min_vis_y, max_vis_y = math.floor(min(py) - margin), math.ceil(max(py) + margin)

    for i in range(min_vis_x, max_vis_x + 1):
        for j in range(min_vis_y, max_vis_y + 1):
            is_hit = (i, j) in visited_mirrors
            circle = patches.Circle((i, j), float(radius), color='gray' if not is_hit else 'red', 
                                    alpha=0.1 if not is_hit else 0.3, zorder=2)
            ax.add_patch(circle)

    ax.set_aspect('equal')
    ax.set_title(f"IEEE {bit_depth} | Bounces: {bounces} | Dist: {final_dist:.8f}")
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend()
    plt.show()

interact(run_ieee_simulation, bit_depth=Dropdown(
    options=['32-bit (Single)', '64-bit (Double)'],
    value='64-bit (Double)',
    description='IEEE Type:'
))

interactive(children=(Dropdown(description='IEEE Type:', index=1, options=('32-bit (Single)', '64-bit (Double)…

<function __main__.run_ieee_simulation(bit_depth)>